## Post Process Filtering

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# case_names = ["run_smoother_udu_r7_pr3_iter5", "run_smoother_udu_r7_pr10_iter5", "run_smoother_udu_r7_pr20_iter5", "run_smoother_udu_r7_pr30_iter5", "run_smoother_udu_if30_iter5"]
# case_names = ["run_smoother_udu_tdcp_if7_iter8", "run_smoother_udu_if7_iter5", "run_smoother_udu_if30_iter5"]
case_names = [
    "sim_pr",
    "sim_ifpr",
    "sim_tdcp",
]
# case_names = ["test_smoother_udu_joseph_tdcp8_if50"]

satid = 0
dt = 1.0
norbit = 6
dt = 1
dt_rt = 120

mcmax = 40  # Max number of Monte Carlo runs

savedir = pnt.get_output_dir() / "Plasmasphere_Delay_Filtering" / "figures"
savedir.mkdir(parents=True, exist_ok=True)

In [ ]:
str_orbitdt = f"norbit_{norbit}_dt_{dt}s_dtrt_{dt_rt}s"

outdirs = {}
for case_name in case_names:
    outdir = (
        pnt.get_output_dir()
        / "Plasmasphere_Delay_Filtering"
        / "v0"
        / "filter"
        / str_orbitdt
        / case_name
    )
    outdirs[case_name] = outdir

In [ ]:
# Load H5 file
import h5py

# list all files in outdir
h5files_filter = {}
h5files_smoother = {}

for case_name in case_names:
    print("Case:", case_name)
    outdir = outdirs[case_name]
    mcdirs = [d for d in outdir.glob("*") if d.is_dir()]
    for d in mcdirs:
        # filename: "filter_iter_{}.h5"
        print("directory:", d)

    h5files_filter[case_name] = {}
    h5files_smoother[case_name] = {}

    # find all folders in outdir
    for dir in mcdirs:
        # if directory starts with "mc"
        if dir.stem.startswith("mc"):
            mcidx = dir.stem.split("mc")[-1]
            if int(mcidx) >= mcmax:
                print("  Skipping MC Index:", int(mcidx))
                continue
            filter_files = [
                f for f in dir.glob("*") if f.suffix == ".h5" and "filter" in f.stem
            ]
            if len(filter_files) == 0:
                print("  No filter files found in", dir)
                continue
            h5files_filter[case_name][int(mcidx)] = {}
            h5files_smoother[case_name][int(mcidx)] = {}
            print("  MC Index:", int(mcidx))
            for i, f in enumerate(dir.glob("*")):
                # if end in .h5
                if f.suffix == ".h5" and "filter" in f.stem:
                    iteridx = f.stem.split("filter_iter_")[-1]
                    print("Loading iteration", iteridx, "file:", f)
                    h5files_filter[case_name][int(mcidx)][int(iteridx)] = h5py.File(
                        f, "r"
                    )
                elif f.suffix == ".h5" and "smooth" in f.stem:
                    iteridx = f.stem.split("smooth_iter_")[-1]
                    print("Loading iteration", iteridx, "file:", f)
                    h5files_smoother[case_name][int(mcidx)][int(iteridx)] = h5py.File(
                        f, "r"
                    )


# List all groups
# for case_name in case_names:
#     for mcidx in h5files_filter[case_name].keys():
#         for iteridx in h5files_filter[case_name][mcidx].keys():
#             print(
#                 f"Filter - Case: {case_name}, MC Index: {mcidx}, Iter Index: {iteridx}"
#             )
#             f = h5files_filter[case_name][mcidx][iteridx]
#             print("  Groups:", list(f.keys()))

#             if iteridx in h5files_smoother[case_name][mcidx]:
#                 fs = h5files_smoother[case_name][mcidx][iteridx]
#                 print(
#                     f"Smoother - Case: {case_name}, MC Index: {mcidx}, Iter Index: {iteridx}"
#                 )
#                 print("  Groups:", list(fs.keys()))

In [ ]:
# Compute Errors
from src_py.filter_postprocess import compute_od_errors

print(h5files_filter[case_name].keys())
print(h5files_smoother[case_name].keys())

smooth_selection = ["best", "last", "first", "best_dz", "5"]
total = ["mean", "cat"]

filter_sise_pos_all = {}
filter_sise_vel_all = {}
smoother_sise_pos_all = {}
smoother_sise_vel_all = {}
filter_sise_pos_mat_all = {}
filter_sise_vel_mat_all = {}
smoother_sise_pos_mat_all = {}
smoother_sise_vel_mat_all = {}

for case_name in case_names:
    print("Case: ", case_name)
    h5_filter = h5files_filter[case_name]
    if len(h5files_smoother[case_name][0]) > 0:
        h5file_smoother = h5files_smoother[case_name]
    else:
        h5file_smoother = []

    print("h5_filter keys:", h5_filter.keys())
    (
        filter_sise_pos_cat,
        filter_sise_vel_cat,
        smoother_sise_pos_cat,
        smoother_sise_vel_cat,
        filter_sise_pos_mat,
        filter_sise_vel_mat,
        smoother_sise_pos_mat,
        smoother_sise_vel_mat,
    ) = compute_od_errors(
        h5_filter,
        h5file_smoother,
        start_ratio=5 / 6,
        end_ratio=6 / 6,
        smooth_selection="last",
        total="cat",
        print_all_cases=False,
    )
    print(" ")
    print(" ")

    filter_sise_pos_all[case_name] = filter_sise_pos_cat
    filter_sise_vel_all[case_name] = filter_sise_vel_cat
    smoother_sise_pos_all[case_name] = smoother_sise_pos_cat
    smoother_sise_vel_all[case_name] = smoother_sise_vel_cat
    filter_sise_pos_mat_all[case_name] = filter_sise_pos_mat
    filter_sise_vel_mat_all[case_name] = filter_sise_vel_mat
    smoother_sise_pos_mat_all[case_name] = smoother_sise_pos_mat
    smoother_sise_vel_mat_all[case_name] = smoother_sise_vel_mat

## Plot histogram

In [ ]:
# Plot the histogram of all the sise errors
figf, axsf = plt.subplots(1, 2, figsize=(10, 4))  # for filters
case_names_plot = ["sim_pr", "sim_ifpr", "sim_tdcp"]
case_labels = ["L1 PR only", "Ionofree PR only", "Ionofree PR + TDCP"]
pos_bins = np.linspace(0, 40, 100)
vel_bins = np.linspace(0, 2.5, 100)
plot_percentile = False
percentile = 99.7
plot_threshold = True
threshold_pos = 13.43  # meters
threshold_vel = 1.2  # m/s
black_bg = False
with_legend = True

colors = ["blue", "orange", "green"]

fontsize_label = 12
fontsize_ticks = 12
fontsize_legend = 12

# filter plots
for i, case_name in enumerate(case_names_plot):
    filter_sise_pos_cat = filter_sise_pos_all[case_name]
    filter_sise_vel_cat = filter_sise_vel_all[case_name]
    pos_prctile = np.percentile(filter_sise_pos_cat, percentile)
    vel_prctile = np.percentile(filter_sise_vel_cat, percentile)
    axsf[0].hist(
        filter_sise_pos_cat,
        bins=pos_bins,
        alpha=0.5,
        color=colors[i],
        label=f"Filter - {case_labels[i]}",
        density=True,
    )
    axsf[1].hist(
        filter_sise_vel_cat,
        bins=vel_bins,
        alpha=0.5,
        color=colors[i],
        label=f"Filter - {case_labels[i]}",
        density=True,
    )

if plot_threshold:
    axsf[0].axvline(
        threshold_pos,
        color="black",
        linestyle=":",
        label=f"Target Threshold: {threshold_pos:.2f} m",
    )
    axsf[1].axvline(
        threshold_vel,
        color="black",
        linestyle=":",
        label=f"Target Threshold: {threshold_vel:.2f} mm/s",
    )
axsf[0].set_title("Position SISE Error Histogram", fontsize=fontsize_label)
axsf[0].set_xlabel("SISE Position Error (m)", fontsize=fontsize_label)
axsf[1].set_title("Velocity SISE Error Histogram", fontsize=fontsize_label)
axsf[1].set_xlabel("SISE Velocity Error (m/s)", fontsize=fontsize_label)

if plot_percentile:
    # compute RMS and add to legend
    for i, case_name in enumerate(case_names_plot):
        filter_sise_pos_cat = filter_sise_pos_all[case_name]
        filter_sise_vel_cat = filter_sise_vel_all[case_name]
        pos_prctile = np.percentile(filter_sise_pos_cat, percentile)
        vel_prctile = np.percentile(filter_sise_vel_cat, percentile)
        # axvline
        axsf[0].axvline(
            pos_prctile,
            color=colors[i],
            linestyle="--",
            # label=f"RMS Filter - {case_labels[i]}: {rms_pos:.2f} m",
        )
        axsf[1].axvline(
            vel_prctile,
            color=colors[i],
            linestyle="--",
            # label=f"RMS Filter - {case_labels[i]}: {rms_vel:.2f} m/s",
        )
for ax in axsf:
    if with_legend:
        ax.legend(fontsize=fontsize_legend)
    ax.grid()
    ax.set_ylabel("Density", fontsize=fontsize_label)
    ax.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
    if black_bg:
        ax.set_facecolor("black")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        ax.tick_params(axis="x", colors="white")
        ax.tick_params(axis="y", colors="white")
        legend = ax.get_legend()
        for text in legend.get_texts():
            text.set_color("white")

if black_bg:
    figf.patch.set_facecolor("black")


plt.tight_layout()
plt.savefig(savedir / "filter_sise_error_histogram.pdf")
plt.show()

# Smoother plots
figs, axss = plt.subplots(1, 2, figsize=(10, 4))  # for smoothers
for i, case_name in enumerate(case_names_plot):
    smoother_sise_pos_cat = smoother_sise_pos_all[case_name]
    smoother_sise_vel_cat = smoother_sise_vel_all[case_name]
    pos_prctile = np.percentile(smoother_sise_pos_cat, percentile)
    vel_prctile = np.percentile(smoother_sise_vel_cat, percentile)

    axss[0].hist(
        smoother_sise_pos_cat,
        bins=pos_bins,
        alpha=0.5,
        color=colors[i],
        label=f"Smoother - {case_labels[i]}",
        density=True,
    )
    axss[1].hist(
        smoother_sise_vel_cat,
        bins=vel_bins,
        alpha=0.5,
        color=colors[i],
        label=f"Smoother - {case_labels[i]}",
        density=True,
    )
if plot_threshold:
    axss[0].axvline(
        threshold_pos,
        color="black",
        linestyle=":",
        label=f"Target Threshold: {threshold_pos:.2f} m",
    )
    axss[1].axvline(
        threshold_vel,
        color="black",
        linestyle=":",
        label=f"Target Threshold: {threshold_vel:.2f} mm/s",
    )
axss[0].set_title("Position SISE Error Histogram", fontsize=fontsize_label)
axss[0].set_xlabel("SISE Position Error (m)", fontsize=fontsize_label)
axss[1].set_title("Velocity SISE Error Histogram", fontsize=fontsize_label)
axss[1].set_xlabel("SISE Velocity Error (m/s)", fontsize=fontsize_label)

if plot_percentile:
    # compute RMS and add to legend
    for i, case_name in enumerate(case_names_plot):
        smoother_sise_pos_cat = smoother_sise_pos_all[case_name]
        smoother_sise_vel_cat = smoother_sise_vel_all[case_name]
        pos_prctile = np.percentile(smoother_sise_pos_cat, percentile)
        vel_prctile = np.percentile(smoother_sise_vel_cat, percentile)
        # axvline
        axss[0].axvline(
            pos_prctile,
            color=colors[i],
            linestyle="--",
            # label=f"RMS Smoother - {case_labels[i]}: {rms_pos:.2f} m",
        )
        axss[1].axvline(
            vel_prctile,
            color=colors[i],
            linestyle="--",
            # label=f"RMS Smoother - {case_labels[i]}: {rms_vel:.2f} m/s",
        )

for ax in axss:
    if with_legend:
        ax.legend(fontsize=fontsize_legend)
    ax.grid()
    ax.set_ylabel("Density", fontsize=fontsize_label)
    ax.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
plt.tight_layout()
plt.savefig(savedir / "smoother_sise_error_histogram.pdf")

plt.show()

### Plot Bar Plot

In [ ]:
# Plot the bar plot of the mean errors for each case

# For filters
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
case_names_plot = ["sim_ifpr", "sim_tdcp"]
case_labels = ["Ionofree PR only", "Ionofree PR + TDCP"]
colors = ["blue", "orange", "green"]
fontsize_label = 12
fontsize_ticks = 12
fontsize_legend = 10
barwidth = 0.25
n_mc = mcmax

for i, case_name in enumerate(case_names_plot):
    filter_sise_pos_rms = filter_sise_pos_mat_all[case_name][:n_mc, 0, 0]  # iter=0, rms
    filter_sise_vel_rms = filter_sise_vel_mat_all[case_name][:n_mc, 0, 0]  # iter=0, rms
    filter_sise_pos_p95 = filter_sise_pos_mat_all[case_name][:n_mc, 0, 1]  # iter=0, p95
    filter_sise_vel_p95 = filter_sise_vel_mat_all[case_name][:n_mc, 0, 1]  # iter=0, p95
    filter_sise_pos_p99 = filter_sise_pos_mat_all[case_name][:n_mc, 0, 2]  # iter=0, p99
    filter_sise_vel_p99 = filter_sise_vel_mat_all[case_name][:n_mc, 0, 2]  # iter=0, p99
    ax = axes[0]
    x = np.arange(n_mc) + i * barwidth - barwidth / 2
    ax.bar(
        x,
        filter_sise_pos_rms,
        width=barwidth,
        alpha=1,
        color=colors[i],
        label=f"Filter - {case_labels[i]}",
    )
    ax = axes[1]
    ax.bar(
        x,
        filter_sise_vel_rms,
        width=barwidth,
        alpha=1,
        color=colors[i],
        label=f"Filter - {case_labels[i]}",
    )
for ax in axes:
    ax.legend(fontsize=fontsize_legend)
    ax.grid()
    ax.set_xlabel("Monte Carlo Run Index", fontsize=fontsize_label)
    ax.set_ylabel("RMS SISE Error", fontsize=fontsize_label)
    ax.tick_params(axis="both", which="major", labelsize=fontsize_ticks)
plt.tight_layout()
plt.savefig(savedir / "filter_sise_error_bar.pdf")

## Plot Convergence Plot

In [ ]:
from src_py.filter_postprocess import plot_errors

caseidx = 2
case_str = case_names[caseidx]
mcidx = 0
iter = 0

ylims = {
    "pos": [-50, 50],
    "pos_norm": [0, 50],
    "vel": [-1, 1],
    "vel_norm": [0, 10],
    "clkb": [-50, 50],
    "clkd": [-0.005, 0.005],
    "clkdd": [-1e-6, 1e-6],
    "srp": [-1e-3, 1e-3],
}

# ylims = {
#     "pos": [-1e6, 1e6],
#     "pos_norm": [0, 50],
#     "vel": [-1000, 1000],
#     "vel_norm": [0, 10],
#     "clkb": [-1e6, 1e6],
#     "clkd": [-0.005, 0.005],
#     "clkdd": [-1e-6, 1e-6],
#     "srp": [-1e-3, 1e-3],
# }

# filter errors
plot_errors(
    h5files_filter[case_str][mcidx][iter],
    plot_inv=50,
    ylims=ylims,
    use_rtn=True,
    is_smoother=False,
    n_orbits=norbit,
    plot_sigma=True,
)
savedir = pnt.get_output_dir() / "Plasmasphere_Delay_Filtering" / "figures"
savedir.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(
    savedir / "filter_errors_case{}_mcidx{}_iter{}.pdf".format(caseidx, mcidx, iter)
)
plt.show()
print(
    "Saved filename: filter_errors_case{}_mcidx{}_iter{}.pdf".format(
        caseidx, mcidx, iter
    )
)

## Plot Smoother Iteration

In [ ]:
# smoother errors
from src_py.filter_postprocess import plot_sise_errors

ylims_sm = {
    "pos": [-10, 10],
    "pos_norm": [0, 10],
    "vel": [-1, 1],
    "vel_norm": [0, 2],
    "clkb": [-10, 10],
    "clkd": [-0.001, 0.001],
    "clkdd": [-1e-7, 1e-7],
    "srp": [-1e-3, 1e-3],
}

case_str = case_names[1]
mcidx = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for i in range(5):
    fig, axes = plot_sise_errors(
        h5files_smoother[case_str][mcidx][i],
        plot_inv=10,
        ylims=ylims_sm,
        use_rtn=True,
        is_smoother=True,
        n_orbits=None,
        fig=fig,
        axes=axes,
        plot_label=f"Iter {i+1}",
        plot_sigma=(i == 0),
    )
plt.tight_layout()
plt.show()

## Plot Filter vs Smoother

In [ ]:
# smoother errors
ylims_sm = {
    "pos": [-20, 20],
    "pos_norm": [0, 20],
    "vel": [-1, 1],
    "vel_norm": [0, 5],
    "clkb": [-20, 20],
    "clkd": [-0.001, 0.001],
    "clkdd": [-1e-7, 1e-7],
    "srp": [-1e-3, 1e-3],
}

caseidx = 2
case_str = case_names[caseidx]
mcidx = 0
iter = 4
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# plot filter
fig, axes = plot_sise_errors(
    h5files_filter[case_str][mcidx][0],
    plot_inv=10,
    ylims=ylims_sm,
    use_rtn=True,
    is_smoother=False,
    n_orbits=None,
    fig=fig,
    axes=axes,
    plot_label=f"Filter",
    plot_sigma=True,
    sigma_color="blue",
)

# plot smoother
fig, axes = plot_sise_errors(
    h5files_smoother[case_str][mcidx][iter],
    plot_inv=10,
    ylims=ylims_sm,
    use_rtn=True,
    is_smoother=True,
    n_orbits=None,
    fig=fig,
    axes=axes,
    plot_label=f"Smoother",
    plot_sigma=True,
    sigma_color="orange",
)

# get savedir
savedir = pnt.get_output_dir() / "Plasmasphere_Delay_Filtering" / "figures"
savedir.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(
    savedir
    / "filter_smoother_comparison_case{}_mcidx{}_iter{}.pdf".format(
        caseidx, mcidx, iter
    )
)

plt.show()